# Cài thư viện

In [2]:
!pip -q install -U transformers accelerate safetensors pillow pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 102.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 96.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 98.1 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.3 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-col

In [4]:
!pip install -U "pillow<12"

# import và cấu hình

In [5]:
import os
import re
import json
from pathlib import Path
from PIL import Image

import torch
import pandas as pd
from transformers import AutoProcessor, Pix2StructForConditionalGeneration


# config đường dẫn

In [6]:
INPUT_DIR = "/kaggle/input/datasets/mainhatthong/test-chart/test"
OUTPUT_DIR = "/kaggle/working/deplot_output"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "raw_text"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "json"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "csv"), exist_ok=True)

MODEL_NAME = "google/deplot"
SUPPORTED_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}

In [8]:
from pathlib import Path

p = Path(INPUT_DIR)
print("Exists:", p.exists())
print("Is dir:", p.is_dir())
print("Contents:", list(p.iterdir())[:10] if p.exists() else "No such folder")


Exists: True
Is dir: True
Contents: [PosixPath('/kaggle/input/datasets/mainhatthong/test-chart/test/bao_viet_holdings_2023_p148_jpg.rf.t6HA9xW3jFvVbQ3iewRq_chart_367.jpg'), PosixPath('/kaggle/input/datasets/mainhatthong/test-chart/test/bao_viet_holdings_2023_p125_jpg.rf.JKABOhqh1YH2eLBdeOm2_chart_192.jpg'), PosixPath('/kaggle/input/datasets/mainhatthong/test-chart/test/vietnam_airlines_jsc_2024_p025_jpg.rf.viJIhZbhqeNOlCVeKDGz_chart_1616.jpg'), PosixPath('/kaggle/input/datasets/mainhatthong/test-chart/test/bao_viet_holdings_2023_p148_jpg.rf.t6HA9xW3jFvVbQ3iewRq_chart_368.jpg'), PosixPath('/kaggle/input/datasets/mainhatthong/test-chart/test/bao_viet_holdings_2023_p136_jpg.rf.3BoRb5ldtExxU9jlPmCu_chart_84.jpg'), PosixPath('/kaggle/input/datasets/mainhatthong/test-chart/test/bao_viet_holdings_2023_p099_jpg.rf.B54zSe2EoumrOZTIX9dH_chart_92.jpg'), PosixPath('/kaggle/input/datasets/mainhatthong/test-chart/test/ssi_securities_corp_2024_p045_jpg.rf.SEwyoQAfZRAhIyHQ6Jg9_chart_1542.jpg'), PosixP

# load model

In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

processor = AutoProcessor.from_pretrained(MODEL_NAME)
model = Pix2StructForConditionalGeneration.from_pretrained(MODEL_NAME)

model = model.to(device)
model.eval()


Device: cuda


preprocessor_config.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/285 [00:00<?, ?it/s]

Pix2StructForConditionalGeneration(
  (encoder): Pix2StructVisionModel(
    (embeddings): Pix2StructVisionEmbeddings(
      (patch_projection): Linear(in_features=768, out_features=768, bias=True)
      (row_embedder): Embedding(4096, 768)
      (column_embedder): Embedding(4096, 768)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): Pix2StructVisionEncoder(
      (layer): ModuleList(
        (0-11): 12 x Pix2StructVisionLayer(
          (attention): Pix2StructVisionAttention(
            (query): Linear(in_features=768, out_features=768, bias=False)
            (key): Linear(in_features=768, out_features=768, bias=False)
            (value): Linear(in_features=768, out_features=768, bias=False)
            (output): Linear(in_features=768, out_features=768, bias=False)
          )
          (mlp): Pix2StructVisionMlp(
            (wi_0): Linear(in_features=768, out_features=2048, bias=False)
            (wi_1): Linear(in_features=768, out_features=2048, bias=False)
 

# helper functions

In [11]:
def normalize_text(x):
    if x is None:
        return ""
    if isinstance(x, str):
        return x.strip()
    return str(x).strip()


def extract_text_from_output(outputs):
    """
    Safe extraction for different output formats.
    """
    if outputs is None:
        return ""
    if isinstance(outputs, str):
        return outputs
    if isinstance(outputs, dict):
        if "generated_text" in outputs:
            return extract_text_from_output(outputs["generated_text"])
        if "text" in outputs:
            return extract_text_from_output(outputs["text"])
        return str(outputs)
    if isinstance(outputs, (list, tuple)):
        if len(outputs) == 0:
            return ""
        return extract_text_from_output(outputs[0])
    return str(outputs)


def try_parse_table_text(text):
    """
    Best-effort parse for markdown/TSV-like table.
    """
    if not text:
        return None

    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if len(lines) < 2:
        return None

    # markdown table
    if any("|" in line for line in lines):
        rows = []
        for line in lines:
            if "|" not in line:
                continue
            parts = [p.strip() for p in line.strip("|").split("|")]
            if len(parts) >= 2:
                rows.append(parts)
        if len(rows) >= 2:
            return rows

    # tab or multi-space
    rows = []
    for line in lines:
        if "\t" in line:
            parts = [p.strip() for p in line.split("\t")]
        else:
            parts = [p.strip() for p in re.split(r"\s{2,}", line) if p.strip()]
        if len(parts) >= 2:
            rows.append(parts)

    if len(rows) >= 2:
        return rows

    return None


# inference cho 1 ảnh

In [12]:
def run_deplot_on_image(image_path):
    image = Image.open(image_path).convert("RGB")

    # Prompt rất quan trọng
    prompt = (
    "Read the chart and output the data as a table. "
    "Preserve labels and numeric values exactly as shown. "
    "Return only the table."
)

    inputs = processor(
        images=image,
        text=prompt,
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=512
        )

    generated_text = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    generated_text = normalize_text(generated_text)
    parsed_table = try_parse_table_text(generated_text)

    return {
        "image_name": Path(image_path).name,
        "model": MODEL_NAME,
        "prompt": prompt,
        "raw_text": generated_text,
        "parsed_table": parsed_table
    }


# chạy toàn bộ folder

In [13]:
image_files = []
for p in Path(INPUT_DIR).iterdir():
    if p.suffix.lower() in SUPPORTED_EXTS:
        image_files.append(p)

image_files = sorted(image_files)
print("Found", len(image_files), "images")

all_results = []

for idx, img_path in enumerate(image_files, start=1):
    try:
        result = run_deplot_on_image(img_path)
        all_results.append(result)

        # save raw text
        raw_path = os.path.join(OUTPUT_DIR, "raw_text", f"{img_path.stem}.txt")
        with open(raw_path, "w", encoding="utf-8") as f:
            f.write(result["raw_text"])

        # save json
        json_path = os.path.join(OUTPUT_DIR, "json", f"{img_path.stem}.json")
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)

        print(f"[{idx}/{len(image_files)}] OK: {img_path.name}")

    except Exception as e:
        err = {
            "image_name": img_path.name,
            "model": MODEL_NAME,
            "error": repr(e)
        }
        err_path = os.path.join(OUTPUT_DIR, "json", f"{img_path.stem}_error.json")
        with open(err_path, "w", encoding="utf-8") as f:
            json.dump(err, f, ensure_ascii=False, indent=2)

        print(f"[{idx}/{len(image_files)}] FAIL: {img_path.name} -> {repr(e)}")


Found 10 images


Arial.TTF: 0.00B [00:00, ?B/s]

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[1/10] OK: bao_viet_holdings_2023_p099_jpg.rf.B54zSe2EoumrOZTIX9dH_chart_92.jpg
[2/10] OK: bao_viet_holdings_2023_p104_jpg.rf.PC9obWuLU9zAMVtpvgHV_chart_273.jpg


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[7/10] OK: bao_viet_holdings_2024_p097_jpg.rf.0uepUYIW99XRTIwCqdmL_chart_688.jpg


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[8/10] OK: bao_viet_holdings_2024_p236_jpg.rf.5WJD7HmXilvZBEfvBXXm_chart_1842.jpg


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[9/10] OK: ssi_securities_corp_2024_p045_jpg.rf.SEwyoQAfZRAhIyHQ6Jg9_chart_1542.jpg
[10/10] OK: vietnam_airlines_jsc_2024_p025_jpg.rf.viJIhZbhqeNOlCVeKDGz_chart_1616.jpg


# lưu tổng hợp JSONL

In [14]:
jsonl_path = os.path.join(OUTPUT_DIR, "results.jsonl")
with open(jsonl_path, "w", encoding="utf-8") as f:
    for item in all_results:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Saved:", jsonl_path)

Saved: /kaggle/working/deplot_output/results.jsonl


In [15]:
with open("/kaggle/working/deplot_output/results.jsonl", "r", encoding="utf-8") as f:
    print(f.read())

{"image_name": "bao_viet_holdings_2023_p099_jpg.rf.B54zSe2EoumrOZTIX9dH_chart_92.jpg", "model": "google/deplot", "prompt": "Read the chart and output the data as a table. Preserve labels and numeric values exactly as shown. Return only the table.", "raw_text": "TITLE |  <0x0A>  | General Insurance | Life Insurance | Investment <0x0A> JOBS | 41% | 31% | 10% <0x0A> COMPLETED 100%<0x0A>THE AUDIT PLAN<0x0A>APPROVED BY<0x0A>THE BOD | 20 | 20% | 9%", "parsed_table": null}
{"image_name": "bao_viet_holdings_2023_p104_jpg.rf.PC9obWuLU9zAMVtpvgHV_chart_273.jpg", "model": "google/deplot", "prompt": "Read the chart and output the data as a table. Preserve labels and numeric values exactly as shown. Return only the table.", "raw_text": "TITLE |  <0x0A> Booviet identified the stakeholders based on the role<0x0A>and level of influence of the parties involved in Booviet<0x0A>proportionally as follows: | Employees: 25% | Media: 15% | Customers: 15% | Shareholders/Investors: 20% | Government: 10% | Loca

In [16]:
import json
import pandas as pd

path = "/kaggle/working/deplot_output/results.jsonl"

rows = []
with open(path, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)
df.head(10)


,image_name,model,prompt,raw_text,parsed_table
0,bao_viet_holdings_2023_p099_jpg.rf.B54zSe2Eoum...,google/deplot,Read the chart and output the data as a table....,TITLE | <0x0A> | General Insurance | Life In...,None
1,bao_viet_holdings_2023_p104_jpg.rf.PC9obWuLU9z...,google/deplot,Read the chart and output the data as a table....,TITLE | <0x0A> Booviet identified the stakeho...,None
2,bao_viet_holdings_2023_p125_jpg.rf.JKABOhqh1YH...,google/deplot,Read the chart and output the data as a table....,TITLE | Entry level wage of newly recruited fe...,None
3,bao_viet_holdings_2023_p136_jpg.rf.3BoRb5ldtEx...,google/deplot,Read the chart and output the data as a table....,TITLE | Investment Portfolio<0x0A>at 31.12.202...,None
4,bao_viet_holdings_2023_p148_jpg.rf.t6HA9xW3jFv...,google/deplot,Read the chart and output the data as a table....,TITLE | Distance traveled in 2023 (km)<0x0A>Ye...,None
5,bao_viet_holdings_2023_p148_jpg.rf.t6HA9xW3jFv...,google/deplot,Read the chart and output the data as a table....,TITLE | Fuel consumption & usage expense of 20...,None
6,bao_viet_holdings_2024_p097_jpg.rf.0uepUYIW99X...,google/deplot,Read the chart and output the data as a table....,TITLE | <0x0A> Liabilities | Liabilities<0x0A...,None
7,bao_viet_holdings_2024_p236_jpg.rf.5WJD7HmXilv...,google/deplot,Read the chart and output the data as a table....,Entity | Value <0x0A> General Insurance: | 15 ...,None
8,ssi_securities_corp_2024_p045_jpg.rf.SEwyoQAfZ...,google/deplot,Read the chart and output the data as a table....,TITLE | <0x0A> | Total <0x0A> Over 50 | 1.61...,None
9,vietnam_airlines_jsc_2024_p025_jpg.rf.viJIhZbh...,google/deplot,Read the chart and output the data as a table....,Country | Revenue passenger - kilometers (RPK)...,None
